# Week 8 · Topic 4 — From Functions to a Class (optional extra)

**Box we are building: the project's real architecture, starting now**

The same refactor Week 5's demo did to `record_check.py`. This time you do it
to your own code.

> **Already know this?** Skip to the *Topic 4 Challenge*.

> **Optional extra.** Your CW2 project uses plain functions, not classes, so you do not need this topic for the lab. It shows how the same code looks when it is bundled into a class.

---
## 1. Before: loose functions and a stray variable

In [1]:
def parse_line(raw_line):
    """Turn one messy raw line into a dict."""
    label, value = raw_line.strip().split(",")
    return {"label": label.strip().lower(), "value": float(value)}

def status_of(value, limit=100):
    """Return OVER LIMIT or OK for a value against a limit."""
    return "OVER LIMIT" if value >= limit else "OK"

over_limit_count = 0   # <- not attached to anything

### Your turn
Call `parse_line` and `status_of` on `"srv-01, 87"` and print the result.

In [2]:
# your code here
record = parse_line("srv-01, 87")
print(record, "->", status_of(record["value"]))

{'label': 'srv-01', 'value': 87.0} -> OK


---
## 2. After: one class, bundling the same data and logic

Nothing here is new — every line is something Week 5 already taught. What
changed is where it lives.

In [3]:
class RecordSource:
    """Loads raw lines, parses them, and tracks how many are over limit."""

    def __init__(self, limit=100):
        self.limit = limit
        self.records = []
        self.over_limit_count = 0

    def parse_line(self, raw_line):
        label, value = raw_line.strip().split(",")
        return {"label": label.strip().lower(), "value": float(value)}

    def status_of(self, value):
        return "OVER LIMIT" if value >= self.limit else "OK"

```
before: loose functions            after: one class

parse_line(raw_line)                class RecordSource:
status_of(value, limit)                 self.limit
over_limit_count  (stray)               self.over_limit_count
                                         def parse_line(self, raw_line): ...
                                         def status_of(self, value): ...

same logic - now it travels together, and two RecordSource objects
never interfere with each other
```

### Your turn
Add a `load(self, raw_lines)` method that parses every line, prints its status, and updates `self.over_limit_count`.

In [4]:
# your code here
def load(self, raw_lines):
    """Parse every line, print its status, and update self.over_limit_count."""
    for raw_line in raw_lines:
        record = self.parse_line(raw_line)
        record["status"] = self.status_of(record["value"])
        self.records.append(record)
        if record["status"] == "OVER LIMIT":
            self.over_limit_count += 1
        print(record["label"], record["value"], record["status"])


RecordSource.load = load        # give the class defined in the box above its new method

source = RecordSource()
source.load(["srv-01, 87", "srv-02, 120", "srv-03, 45"])
print("over limit:", source.over_limit_count)

srv-01 87.0 OK
srv-02 120.0 OVER LIMIT
srv-03 45.0 OK
over limit: 1


---
## 3. What actually changed

`over_limit_count` can no longer be forgotten or reset by accident — it
belongs to the object. `parse_line` and `status_of` no longer need their
inputs passed between calls — they share `self`. You can now make a *second*
`RecordSource` with a different limit, and the two will not interfere.

In [5]:
a = RecordSource(limit=100)
b = RecordSource(limit=50)
print(a.limit, b.limit)   # independent

100 50


### Your turn
Create two `RecordSource` objects with different limits and prove they do not share state.

In [6]:
# your code here
strict = RecordSource(limit=50)
loose = RecordSource(limit=100)

strict.load(["srv-01, 60"])
loose.load(["srv-01, 60"])

print(strict.limit, "->", strict.over_limit_count, "over limit")
print(loose.limit, "->", loose.over_limit_count, "over limit")
print("shared list?", strict.records is loose.records)

srv-01 60.0 OVER LIMIT
srv-01 60.0 OK
50 -> 1 over limit
100 -> 0 over limit
shared list? False


---
## Topic 4 Challenge

Take your Week 7 `parse_line`/`status_of` functions (or the versions above)
and refactor them into one class, exactly the way this topic did — this is
literally this week's project task, rehearsed here first.

In [7]:
# your code here
class SurveySource:
    """Week 7's survey parsing, bundled into one object."""

    def __init__(self, flag_below=3):
        self.flag_below = flag_below
        self.responses = []
        self.flagged_count = 0

    def parse_line(self, raw_line):
        survey_part, rest = raw_line.strip().split(",", 1)
        fields = {}
        for piece in rest.split(","):
            key, value = piece.split("=")
            fields[key.strip().lower()] = float(value.strip())
        return {"survey": survey_part.strip().lower(),
                "id": int(fields["id"]),
                "rating": fields["rating"],
                "age": fields["age"]}

    def status_of(self, rating):
        return "FLAGGED" if rating < self.flag_below else "OK"

    def load(self, raw_lines):
        for raw_line in raw_lines:
            record = self.parse_line(raw_line)
            record["status"] = self.status_of(record["rating"])
            self.responses.append(record)
            if record["status"] == "FLAGGED":
                self.flagged_count += 1
            print(record["id"], record["rating"], record["status"])


raw_lines = [
    "  Survey_2026 , id=101 , rating=4 , age=23",
    "SURVEY_2026,ID=102,RATING=2,AGE=31",
    "survey_2026 , id=103 , rating=5 , age=19",
]

survey = SurveySource()
survey.load(raw_lines)
print(survey.flagged_count, "flagged of", len(survey.responses))

101 4.0 OK
102 2.0 FLAGGED
103 5.0 OK
1 flagged of 3


---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | a `Dataset` class bundling loading, cleaning and summarising |
| **Cyber Security** | an `IncidentLog` class bundling loading, parsing and flagging |
| **IT** | an `OpsConsole` class bundling fetching, checking and reporting |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| forgetting to change `value` to `self.value` when moving a function in | `NameError` or wrong result | every reference to shared data needs `self.` |
| leaving `limit` as a function parameter instead of `self.limit` | works, but defeats the point — nothing is bundled | store it in `__init__`, read it via `self` |
| one class doing everything with no clear responsibility | hard to test, hard to extend | keep it to the same "one job" rule as functions had |
| reaching for inheritance or a second class | more complexity than this project needs yet | one or two classes is the whole ask this week |

## Go deeper

`examples/01_before_and_after.py` — the exact refactor, side by side
`examples/02_a_class_that_pulls_real_data.py` — the same idea as functions that share settings (optional topic: the lab itself uses plain functions)

---
## Recap

| You can now | Using |
|---|---|
| spot when functions should become a class | shared data, passed around too much |
| do the refactor | bundle data in `__init__`, methods use `self` |
| keep two instances independent | separate objects, separate state |

**You now have all of Week 8's tools.** This is what starts your project in
the lab. The project you start there is the one you keep building until Week 12.